1. Make an artificial DNA sequence of 200-400b in length, in which to simulate 3-4 transposable elements.

In [1]:
import random

random.seed(7)
COMP = str.maketrans("ACGT", "TGCA")

def revcomp(s):
    return s.translate(COMP)[::-1]

def rand_dna(n):
    return "".join(random.choice("ACGT") for _ in range(n))

IR_LEN, TSD_LEN = 8, 5

def make_transposon():
    ir = rand_dna(IR_LEN)
    return ir + rand_dna(random.randint(18, 30)) + revcomp(ir)

def insert_transposon(seq, elements, pos):
    te = make_transposon()
    tsd = seq[pos:pos + TSD_LEN]
    new = seq[:pos] + tsd + te + seq[pos:]
    start = pos + TSD_LEN
    end = start + len(te) - 1
    shift = TSD_LEN + len(te)
    updated = []
    for s, e in elements:
        if pos + 0 <= s:
            s, e = s + shift, e + shift
        elif pos <= e:
            e += shift
        updated.append((s, e))
    updated.append((start, end))
    return new, updated

seq = rand_dna(120)
true_elements = []

seq, true_elements = insert_transposon(seq, true_elements, 30)
seq, true_elements = insert_transposon(seq, true_elements, 100)

inside_te1 = true_elements[0][0] + IR_LEN + 8
seq, true_elements = insert_transposon(seq, true_elements, inside_te1)

inside_te2 = true_elements[1][0] + IR_LEN + 6
seq, true_elements = insert_transposon(seq, true_elements, inside_te2)

assert 200 <= len(seq) <= 400
print("Length:", len(seq))
print(seq)
print("\nGround truth (0-based, inclusive):")
for i, (s, e) in enumerate(sorted(true_elements), 1):
    print(f"  TE{i}: start={s}, end={e}, len={e - s + 1}")

Length: 304
GCTAAAGACAATTACATAACATACACGTCAGCACGTGGCATTTTATTACACTCAGACCTGAAGTCGTGGACACTCGCTATGAATCTACTTCAGGTCAGAAACAGAACTCGAAATGCCAGCACGAAACTTGTTGGCCCAGTGTGAATCGGGTAATTTGACAGGTCACGCTGATTTACCCACTCTGCCAAACTCCAGCGCGGTCAGTAAATCAGTCACGCAGAGGCGCGCCCTAAATTACCAATCGCTTAAGGGTTAAGTAAGTGTGATGCATACGCCTTTACTTGCTGTGTCCACCCCATCGGAC

Ground truth (0-based, inclusive):
  TE1: start=35, end=117, len=83
  TE2: start=56, end=93, len=38
  TE3: start=148, end=238, len=91
  TE4: start=167, end=211, len=45


2. Implement a software application to detect the positions of these transposable elements (start, end) within the created DNA sequence.

In [2]:
def find_transposons(seq, ir_len=IR_LEN, tsd_len=TSD_LEN, min_len=30, max_len=200):
    found = []
    n = len(seq)
    for i in range(tsd_len, n - ir_len):
        left_ir = seq[i:i + ir_len]
        rc = revcomp(left_ir)
        j = seq.find(rc, i + min_len - ir_len)
        while j != -1 and j + ir_len - i <= max_len:
            end = j + ir_len - 1
            if end + tsd_len < n and seq[i - tsd_len:i] == seq[end + 1:end + 1 + tsd_len]:
                found.append((i, end))
            j = seq.find(rc, j + 1)
    return found

detected = sorted(find_transposons(seq))
print("Detected transposable elements (0-based, inclusive):")
for i, (s, e) in enumerate(detected, 1):
    print(f"  #{i}: start={s}, end={e}, len={e - s + 1}")

print("\nGround truth      :", sorted(true_elements))
print("Detected          :", detected)
print("All found correctly:", set(true_elements) <= set(detected))

for s, e in detected:
    parents = [(s2, e2) for s2, e2 in detected if s2 < s and e < e2]
    if parents:
        print(f"Element ({s}, {e}) is nested inside {parents}")

Detected transposable elements (0-based, inclusive):
  #1: start=35, end=117, len=83
  #2: start=56, end=93, len=38
  #3: start=148, end=238, len=91
  #4: start=167, end=211, len=45

Ground truth      : [(35, 117), (56, 93), (148, 238), (167, 211)]
Detected          : [(35, 117), (56, 93), (148, 238), (167, 211)]
All found correctly: True
Element (56, 93) is nested inside [(35, 117)]
Element (167, 211) is nested inside [(148, 238)]


Download a fasta file from NCBI that contains an entire genome (bacterial/plant, something up to 6 MB). Change your application such that it may find new transposons across the entire genome. 

Note: the inverted repeates this time are unknown. 

Proposal: one way to solve this task would be to use two different sliding windows S1 and S2. The first sliding window S1 should slide across the entire genome with a step of 1. At each new position for S1, the S2 sliding window should search for the second pattern up to the end of the genome. Note that the second pattern should be the content found in S1, inverted and what you have to search is the complementary part of this inversion.

Note: the minimum length between S1 and S2 must be at least 100 bases.

Note: the sizes of both windows should be between 5 and 8. The sliding windows should pass multiple times over the genome, once when S1 and S2 are of length 5, the second time when they are of length 6 and so on until 8.

In [3]:
import time
import numpy as np

# 1. load the genome (E. coli CFT073, NC_004431.1, ~5.2 Mb) from the FASTA file
def read_fasta(path):
    with open(path) as f:
        lines = [l.strip() for l in f]
    return lines[0], "".join(lines[1:]).upper()

header, genome = read_fasta("ecoli_cft073.fasta")
print(header)
print("Genome length:", len(genome), "bases")

# 2. parameters
MIN_DIST = 100          # minimum distance between S1 and S2
MAX_DIST = 5000         # maximum transposon size searched (known IS elements are < ~5 kb)
WINDOW_SIZES = range(5, 9)   # S1 and S2 have the same size, one pass over the genome per size: 5, 6, 7, 8
MIN_IR = 16             # windows of 5-8 bases match by pure chance almost everywhere, so
                        # neighbouring matches are merged and only inverted repeats >= MIN_IR are kept

# 3. helpers: every window is encoded as an integer (2 bits / base), so comparing two windows
#    is one integer comparison and S2 can be compared with every S1 in a vectorised way
def encode_kmers(seq, k):
    """Integer code of every k-mer (step 1); -1 if the window contains N or another symbol."""
    lut = np.full(256, -1, dtype=np.int32)
    for code, b in enumerate("ACGT"):
        lut[ord(b)] = code
    x = lut[np.frombuffer(seq.encode(), dtype=np.uint8)]
    n = len(x) - k + 1
    codes = np.zeros(n, dtype=np.int32)
    bad = np.zeros(n, dtype=bool)
    for t in range(k):
        part = x[t:t + n]
        bad |= part < 0
        codes = (codes << 2) | np.where(part < 0, 0, part)
    codes[bad] = -1
    return codes

def revcomp_codes(codes, k):
    """Code of the reverse complement (inverted + complemented) of each encoded k-mer."""
    out = np.zeros_like(codes)
    c = codes.copy()
    for _ in range(k):
        out = (out << 2) | (3 - (c & 3))
        c >>= 2
    out[codes < 0] = -2          # -2 never equals a valid code nor the -1 marker of fwd
    return out

def one_pass(k):
    """S1 slides over the genome (step 1); S2 looks, downstream of every S1 and at a distance of
    MIN_DIST..MAX_DIST, for the reverse complement of S1. Returns the (i, j) pairs found."""
    fwd = encode_kmers(genome, k)       # content of S1 at every position
    rev = revcomp_codes(fwd, k)         # the pattern S2 has to find
    n = len(fwd)
    I, J = [], []
    for d in range(MIN_DIST, MAX_DIST + 1):          # d = j - i
        idx = np.nonzero(fwd[:n - d] == rev[d:])[0]
        I.append(idx.astype(np.int32))
        J.append((idx + d).astype(np.int32))
    return np.concatenate(I), np.concatenate(J)

def merge_pairs(I, J, k):
    """Windows i, i+1, ... paired with j, j-1, ... (constant i + j) are one longer inverted repeat."""
    key = np.sort((I.astype(np.int64) + J) << 24 | I)     # sort by diagonal (i+j), then by i
    s, i = key >> 24, key & ((1 << 24) - 1)
    new_run = np.ones(len(key), dtype=bool)
    new_run[1:] = (s[1:] != s[:-1]) | (i[1:] != i[:-1] + 1)
    first = np.nonzero(new_run)[0]
    last = np.append(first[1:], len(key)) - 1
    start = i[first]
    end = s[first] - i[first] + k - 1                      # j of the first window + k - 1
    ir_len = i[last] - i[first] + k
    return start, end, ir_len

# 4. run the passes
found = {}          # (start, end) -> (inverted repeat length, window sizes that detected it)
for k in WINDOW_SIZES:
    t0 = time.time()
    I, J = one_pass(k)
    start, end, ir_len = merge_pairs(I, J, k)
    keep = ir_len >= MIN_IR
    print(f"window size {k}: {len(I):>10,} S1/S2 matches -> {len(start):>9,} merged repeats "
          f"-> {keep.sum():>3} with inverted repeat >= {MIN_IR}   ({time.time() - t0:.0f} s)")
    for s, e, l in zip(start[keep], end[keep], ir_len[keep]):
        prev = found.get((int(s), int(e)), (0, []))
        found[(int(s), int(e))] = (max(prev[0], int(l)), prev[1] + [k])

# 5. report
print(f"\nCandidate transposons found: {len(found)}")
print(f"{'#':>3} {'start':>9} {'end':>9} {'length':>7} {'IR len':>7} windows   inverted repeat (S1)")
for n, ((s, e), (l, ks)) in enumerate(sorted(found.items()), 1):
    print(f"{n:>3} {s:>9} {e:>9} {e - s + 1:>7} {l:>7} {ks}   {genome[s:s + l]}")


>NC_004431.1 Escherichia coli CFT073, complete sequence
Genome length: 5231428 bases


window size 5: 30,738,798 S1/S2 matches -> 22,442,873 merged repeats ->  48 with inverted repeat >= 16   (434 s)


window size 6:  8,299,319 S1/S2 matches -> 6,048,064 merged repeats ->  48 with inverted repeat >= 16   (24 s)


window size 7:  2,252,177 S1/S2 matches -> 1,638,909 merged repeats ->  48 with inverted repeat >= 16   (24 s)


window size 8:    613,544 S1/S2 matches ->   445,319 merged repeats ->  48 with inverted repeat >= 16   (24 s)

Candidate transposons found: 48
  #     start       end  length  IR len windows   inverted repeat (S1)
  1    115304    116778    1475      16 [5, 6, 7, 8]   CTGATGCGACGCTGGC
  2    119511    123314    3804      16 [5, 6, 7, 8]   CGGCCTGCGTTCAGCT
  3    235056    236771    1716      18 [5, 6, 7, 8]   TATCAGACAATCTGTGTG
  4    414129    414406     278      16 [5, 6, 7, 8]   TGATTCAAAATGTCAG
  5   1135274   1137703    2430      18 [5, 6, 7, 8]   CCAGTTTATTAATGCCGT
  6   1248742   1253131    4390      16 [5, 6, 7, 8]   CAGTCACTTCATTTAT
  7   1309298   1310965    1668      17 [5, 6, 7, 8]   TCTTATCAGGCCTACCG
  8   1312607   1316058    3452      16 [5, 6, 7, 8]   AAACTGCTGCAAAGTG
  9   1387548   1388316     769      25 [5, 6, 7, 8]   CCCGTTACGGGGCGGCGACCTCGCG
 10   1539367   1540547    1181      36 [5, 6, 7, 8]   AAACGGCCCCAGAAGGGGTAATGCCAGTCAGTTAAG
 11   1637167   1640194    3028